# Deduplication

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-07-deduplication.ipynb)

Duplicates arrive from retries, replayed messages, late updates and dirty source systems. Deduplication is
not one operation: first decide what counts as "the same record" (the **key**), then decide which copy wins
(the **rule**). Without a rule that is deterministic, the answer can change between runs.

Previous: [window functions](sql-06-window-functions.ipynb). Next: [time-series analysis](sql-08-time-series.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## Find duplicates first

`GROUP BY key HAVING COUNT(*) > 1` lists the duplicated keys. Normalise the key before grouping when the source
is inconsistent: `' Ava@X.com '` and `'ava@x.com'` are one mailbox.

In [ ]:
create_view(
    "contacts",
    "id INT, email STRING",
    [(1, "ava@x.com"), (2, "ben@x.com"), (3, "ava@x.com"), (4, " Ava@X.com ")],
)

In [ ]:
check(
    "SELECT email, COUNT(*) AS copies FROM contacts GROUP BY email HAVING COUNT(*) > 1",
    [("ava@x.com", 2)],
    "exact duplicates",
)
check(
    """
    SELECT lower(trim(email)) AS normalised, COUNT(*) AS copies
    FROM contacts
    GROUP BY lower(trim(email))
    HAVING COUNT(*) > 1
    """,
    [("ava@x.com", 3)],
    "duplicates after normalising the key",
)

## Keep the latest version of each record

`profile_updates` is an append-only feed: several versions per customer. Keep the newest one with `ROW_NUMBER`.
Customer 2 has two versions with the same `updated_at`, so `update_id` breaks the tie deterministically: the
higher id wins.

In [ ]:
create_view(
    "profile_updates",
    "update_id INT, customer_id INT, email STRING, updated_at DATE",
    [
        (1, 1, "old@x.com", date(2023, 1, 1)),
        (2, 1, "new@x.com", date(2023, 6, 1)),
        (3, 1, "mid@x.com", date(2023, 3, 1)),
        (4, 2, "b@x.com", date(2023, 2, 1)),
        (5, 2, "b2@x.com", date(2023, 2, 1)),
    ],
)

In [ ]:
check(
    """
    SELECT customer_id, update_id, email
    FROM (
        SELECT customer_id, update_id, email,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, update_id DESC) AS rn
        FROM profile_updates
    )
    WHERE rn = 1
    ORDER BY customer_id
    """,
    [(1, 2, "new@x.com"), (2, 5, "b2@x.com")],
    "latest version per customer, deterministic tie-break",
)

### The `GROUP BY MAX` pitfall

The common alternative joins back to the maximum timestamp. It is correct only when the maximum is unique.
Here customer 2 has two rows at the maximum, so the join keeps both.

In [ ]:
check(
    """
    SELECT p.customer_id, p.email
    FROM profile_updates p
    JOIN (
        SELECT customer_id, MAX(updated_at) AS latest
        FROM profile_updates
        GROUP BY customer_id
    ) m ON m.customer_id = p.customer_id AND m.latest = p.updated_at
    ORDER BY p.customer_id, p.email
    """,
    [(1, "new@x.com"), (2, "b2@x.com"), (2, "b@x.com")],
    "GROUP BY MAX keeps both tied rows",
)

## Keep the survivors

Spark temporary views are read-only, so you cannot `DELETE` duplicates from them. The equivalent is to select
the survivors: a row survives when no strictly newer version of the same customer exists. On a real table you
would run a `DELETE` or a `MERGE` (see below) with the same predicate.

In [ ]:
check(
    """
    SELECT p.update_id
    FROM profile_updates p
    WHERE NOT EXISTS (
        SELECT 1 FROM profile_updates k
        WHERE k.customer_id = p.customer_id
          AND (k.updated_at > p.updated_at
               OR (k.updated_at = p.updated_at AND k.update_id > p.update_id))
    )
    ORDER BY p.update_id
    """,
    [(2,), (5,)],
    "survivors: one row per customer",
)

## Apply updates to a base table

Updating a base table from a feed is a join plus a fallback. Take the latest update per key, then use `COALESCE`
so keys without an update keep their base value. Forgetting the fallback is how a whole column gets silently
nulled.

In [ ]:
create_view(
    "limits",
    "customer_id INT, credit_limit INT",
    [(1, 1000), (2, 2000), (3, 1500), (4, 500), (5, 800), (6, 900), (7, 300), (8, 400)],
)
create_view(
    "limit_updates",
    "customer_id INT, credit_limit INT, updated_at DATE",
    [
        (1, 1200, date(2023, 5, 1)),
        (1, 1500, date(2023, 8, 1)),
        (4, 700, date(2023, 4, 1)),
        (7, 350, date(2023, 2, 1)),
    ],
)

In [ ]:
check(
    """
    WITH latest AS (
        SELECT customer_id, credit_limit
        FROM (
            SELECT customer_id, credit_limit,
                   ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) AS rn
            FROM limit_updates
        )
        WHERE rn = 1
    )
    SELECT b.customer_id, COALESCE(l.credit_limit, b.credit_limit) AS credit_limit
    FROM limits b
    LEFT JOIN latest l ON l.customer_id = b.customer_id
    ORDER BY b.customer_id
    """,
    [(1, 1500), (2, 2000), (3, 1500), (4, 700), (5, 800), (6, 900), (7, 350), (8, 400)],
    "latest update applied, base kept elsewhere",
)

## On a table that supports it

Spark temporary views cannot be modified, but tables in Delta Lake or Apache Iceberg can. There the same
logic is a single `MERGE`, run after the source has been deduplicated to one row per key. This cell is a
reference only, because it needs a table format that is not part of this notebook:

```sql
MERGE INTO limits AS target
USING latest_limit_updates AS source
  ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET target.credit_limit = source.credit_limit
WHEN NOT MATCHED THEN INSERT (customer_id, credit_limit) VALUES (source.customer_id, source.credit_limit)
```

## Takeaways

- Define the key, then define a deterministic winner. Add a unique tie-breaker.
- Normalise keys before comparing them.
- "Delete duplicates" means "keep survivors". Express it as a predicate you can check before you run it.
- Guard updates with a fallback so untouched keys keep their values.